# OPDC Subgroup Metadata Preparation

This notebook builds the OPDC equivalent of `subgroups.csv` — a
per-patient-per-visit metadata table (subgroup label, gender, age at
visit) that other OPDC notebooks (e.g. the OPDC data-prep notebook) can
join against, mirroring the PPMI `subgroups.csv` schema.

**What this notebook does:**
1. Loads the OPDC subgroup assignments and renames columns to match the
   PPMI naming convention (`PATNO`, `Group`).
2. Loads and cleans gender from the OPDC baseline table, and merges it in.
3. Loads and cleans age-at-visit from the OPDC longitudinal table, and
   merges it in.
4. Writes the combined table to `data/02_processed/OPDC/subgroups.csv`.
5. Ends with a couple of sanity-check cells inspecting previously
   processed output.

**Note:** unlike `03_OPDC_Data_Prep.ipynb` (which relabels the same raw
`subgroups.csv` codes 0/1/2 to letters A/B/C), this notebook keeps the
`Group` column as the original raw numeric codes. If downstream code
expects letter labels, make sure it's reading the *data-prep* notebook's
in-memory relabeled version rather than this notebook's CSV output.


In [58]:
import pandas as pd
import pickle

### Load Subgroup Assignments

In [59]:
subgroups = pd.read_csv('./../../data/01_raw/OPDC/subgroups.csv')
last_levo = pd.read_csv('./../../data/01_raw/OPDC/last_levodopa.csv' , index_col=0)

In [60]:
# Rename to match the PPMI subgroup-table column convention (PATNO, Group).
# Note: 'Group' here keeps the raw 0/1/2 coding — it is not relabeled to
# A/B/C letters in this notebook (see note above).
subgroups.rename(columns={'OPDC participant_id': 'PATNO', 'subgroup': 'Group'}, inplace=True)

In [62]:
subgroups

,PATNO,Group
0,AH002,0
1,AH007,0
2,AH026,0
3,AH032,0
4,AH046,0
...,...,...
850,WP036,2
851,WP040,2
852,WP041,2
853,WP047,2


In [63]:
# Sanity check: proportion of patients in each subgroup.
subgroups['Group'].value_counts() / subgroups.shape[0]

Group
1    0.521637
0    0.272515
2    0.205848
Name: count, dtype: float64

### Add Gender

In [64]:
gender = pd.read_csv('./../../data/01_raw/OPDC/OPDC_Discovery_PD_arm_baseline.csv')[['subjid', 'gender']]

# OPDC subject IDs are stored as 'site/subjectnumber' (e.g. 'OX/1234');
# strip the site prefix so IDs match the 'PATNO' values used elsewhere.
gender['PATNO'] = [''.join(i.split('/')[1:]) for i in gender['subjid']]
gender.drop(columns='subjid', inplace=True)

# Recode to PPMI's numeric GENDER convention (1 = male, 0 = female).
gender['gender'] = gender['gender'].replace('male', 1)
gender['gender'] = gender['gender'].replace('female', 0)
gender.rename(columns={'gender': 'GENDER'}, inplace=True)

/var/folders/5v/vtmbr3255f59jlqhnxc_l2440000gp/T/ipykernel_90094/529848562.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  gender['gender'] = gender['gender'].replace('female', 0)


In [65]:
gender

,GENDER,PATNO
0,1,AH002
1,0,AH003
2,0,AH004
3,0,AH005
4,0,AH007
...,...,...
977,0,WP054
978,1,WP056
979,1,WP058
980,0,WP059


In [66]:
subgroups = pd.merge(subgroups, gender, on='PATNO')
subgroups

,PATNO,Group,GENDER
0,AH002,0,1
1,AH007,0,0
2,AH026,0,1
3,AH032,0,1
4,AH046,0,1
...,...,...,...
850,WP036,2,1
851,WP040,2,1
852,WP041,2,1
853,WP047,2,1


### Add Age at Visit

In [49]:
age = pd.read_csv('./../../data/01_raw/OPDC/OPDC_Discovery_PD_arm_longitudinal.csv')[['subjid', 'visit', 'age']]

# Same site-prefix stripping as above, to align 'PATNO' across sources.
age['PATNO'] = [''.join(i.split('/')[1:]) for i in age['subjid']]
age.drop(columns='subjid', inplace=True)
age.rename(columns={'age': 'AGE_AT_VISIT', 'visit': 'EVENT_ID'}, inplace=True)

/var/folders/5v/vtmbr3255f59jlqhnxc_l2440000gp/T/ipykernel_90094/3280475754.py:1: DtypeWarning: Columns (354,355,356,380,381,382) have mixed types. Specify dtype option on import or set low_memory=False.
  age = pd.read_csv('./../../data/01_raw/OPDC/OPDC_Discovery_PD_arm_longitudinal.csv')[['subjid', 'visit', 'age']]


In [50]:
# Merging on PATNO alone (not PATNO + EVENT_ID) broadcasts every visit's
# age against every (PATNO, Group, GENDER) row, producing one output row
# per patient-visit — this is what expands the table from one row per
# patient to one row per patient-visit.
subgroups = pd.merge(subgroups, age, on='PATNO')
subgroups

,PATNO,Group,GENDER,EVENT_ID,AGE_AT_VISIT
0,AH002,0,1,1,69.1
1,AH002,0,1,2,70.6
2,AH002,0,1,3,72.3
3,AH002,0,1,4,73.9
4,AH002,0,1,5,75.5
...,...,...,...,...,...
3494,WP047,2,1,5,69.9
3495,WP047,2,1,6,71.8
3496,WP058,2,1,1,53.7
3497,WP058,2,1,2,55.5


In [51]:
subgroups = subgroups.sort_values(["PATNO", "EVENT_ID"]).reset_index(drop=True)

# Years elapsed since the previous visit for this subject (NaN on each
# subject's first visit, since there's no prior visit to diff against).
subgroups["visit_interval_years"] = (subgroups.groupby("PATNO")["AGE_AT_VISIT"].diff())

# Cumulative years since the subject's first visit: treat the first
# visit's undefined interval as 0, then cumulatively sum the intervals.
subgroups["time_since_first_visit"] = (
    subgroups.groupby("PATNO")["visit_interval_years"].transform(lambda x: x.fillna(0).cumsum())
)

In [52]:
subgroups = subgroups.drop(columns = ['EVENT_ID' , 'visit_interval_years']).rename(columns={'time_since_first_visit' : 'EVENT_ID'})

In [53]:
subgroups

,PATNO,Group,GENDER,AGE_AT_VISIT,EVENT_ID
0,AH002,0,1,69.1,0.0
1,AH002,0,1,70.6,1.5
2,AH002,0,1,72.3,3.2
3,AH002,0,1,73.9,4.8
4,AH002,0,1,75.5,6.4
...,...,...,...,...,...
3494,WP060,0,1,72.0,0.0
3495,WP060,0,1,73.5,1.5
3496,WP060,0,1,75.4,3.4
3497,WP060,0,1,76.8,4.8


In [56]:
subgroups = pd.merge(subgroups , last_levo , on=['PATNO' , 'EVENT_ID'] , how='left')

### Export

In [57]:
subgroups.to_csv('./../../data/02_processed/OPDC/subgroups.csv')

### Sanity Check: Inspect Previously Processed Output

**Note:** this reads `P1_MDSUPDRS.pkl`, which doesn't match the filenames the OPDC data-prep notebook actually writes (`P1ON_MDSUPDRS.pkl` / `P1OFF_MDSUPDRS.pkl`). This cell will error unless a plain `P1_MDSUPDRS.pkl` was created some other way (e.g. manually, or by an earlier version of the pipeline) — worth double-checking before relying on it.